# Fundamentals 00.4 - Runtime vLLM Provider API

Objetivo: demostrar `vllm-runtime` como provider OpenAI-compatible para Agentic Systems, usando una ruta all-in-one que puede levantar un endpoint vLLM en Colab/GPU cuando se necesita.

La arquitectura del notebook queda separada por capas:

```text
Unsloth / vLLM setup opcional
    -> servidor OpenAI-compatible en http://127.0.0.1:8000/v1
    -> health check /models
    -> OpenAI SDK smoke
    -> Agentic Systems runtime(provider="vllm-runtime")
```

Regla de diseno: Agentic Systems no administra CUDA ni entrena modelos; consume un endpoint compatible con OpenAI. Este notebook puede preparar ese endpoint para Colab, pero lo hace como infraestructura opcional y con gates claros.

Preferencia de este tutorial:

```text
GPU target: T4/L4
Modelo base: Qwen >= 3, default unsloth/Qwen3-0.6B
Inferencia: vLLM OpenAI-compatible endpoint
Unsloth: camino futuro para 4-bit, fast inference, fine-tuning y export
```


## 0) Instalacion opcional

En notebooks usa `%pip` o `subprocess` sobre `sys.executable`, no `%%python -m pip`.

Este notebook separa dependencias:

- `agentic-systems[openai]`: cliente OpenAI-compatible que usa `vllm-runtime`.
- `unsloth`: utilidad para modelos/futuro fine-tuning y fast inference.
- `vllm`: servidor GPU OpenAI-compatible. La guia de Unsloth recomienda instalarlo con `uv pip install -U vllm --torch-backend=auto` para NVIDIA.

Por default `RUN_INSTALL=False`; act?valo solo en Colab o en un entorno nuevo.


In [ ]:
RUN_INSTALL = False

if RUN_INSTALL:
    import subprocess
    import sys

    commands = [
        [sys.executable, "-m", "pip", "install", "-U", "pip"],
        [sys.executable, "-m", "pip", "install", "-U", "agentic-systems[openai]", "unsloth", "uv", "huggingface_hub", "openai"],
        [sys.executable, "-m", "uv", "pip", "install", "-U", "vllm", "--torch-backend=auto"],
    ]
    for cmd in commands:
        print("$", " ".join(cmd))
        subprocess.check_call(cmd)
else:
    print("RUN_INSTALL=False. Si estas en Colab limpio, activa RUN_INSTALL=True y reinicia runtime despues de instalar.")


## 1) Imports y diagnostico de entorno

`vllm installed=True` solo significa que el paquete existe. Para evitar que un import CUDA roto detenga el notebook, la version de `vllm` se lee con `importlib.metadata`.


In [ ]:
from __future__ import annotations

import gc
import importlib.util
import json
import os
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path
from typing import Optional
from urllib.request import Request

import agentic_systems as toolkit


def safe_package_version(name: str) -> str | None:
    try:
        return package_version(name)
    except PackageNotFoundError:
        return None


def environment_snapshot() -> dict:
    snapshot = {
        "python": sys.executable,
        "agentic_systems": getattr(toolkit, "__version__", "unknown"),
        "vllm_installed": importlib.util.find_spec("vllm") is not None,
        "vllm_version": safe_package_version("vllm"),
        "unsloth_installed": importlib.util.find_spec("unsloth") is not None,
        "unsloth_version": safe_package_version("unsloth"),
        "transformers_version": safe_package_version("transformers"),
        "tokenizers_version": safe_package_version("tokenizers"),
        "torch_version": safe_package_version("torch"),
    }
    try:
        import torch
        snapshot.update({
            "torch_cuda": torch.version.cuda,
            "cuda_available": torch.cuda.is_available(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        })
    except Exception as exc:
        snapshot["torch_error"] = str(exc)
    return snapshot


toolkit.show(environment_snapshot(), title="Environment snapshot")


## 2) Configuracion vLLM y Agentic Systems

Variables usadas:

| Variable | Uso |
|---|---|
| `VLLM_BASE_URL` | Endpoint OpenAI-compatible. Default `http://127.0.0.1:8000/v1`. |
| `VLLM_MODEL` | Modelo servido. Default `unsloth/Qwen3-0.6B`. |
| `VLLM_API_KEY` | API key local, normalmente `EMPTY`. |
| `VLLM_MODE` | Perfil `FAST`, `MEDIUM` o `POWER`. |

Para T4 empieza con `FAST`. Para L4 puedes subir a `MEDIUM` o `POWER` despu?s de validar `/models`.

Modelos recomendados:

| Modelo | Perfil T4 | Uso |
|---|---|---|
| `unsloth/Qwen3-0.6B` | Seguro | Default para validar endpoint, tools y runtime. |
| `unsloth/Qwen3-4B-Instruct-2507` | Posible en 4-bit / contexto corto | Mejor calidad no-thinking, tool/instruction use. |
| `unsloth/Qwen3-4B-Thinking-2507` | Experimental en T4 | Razonamiento; requiere contexto mayor y m?s VRAM. Mejor en L4/A100. |

Para T4, empieza con `FAST`, `max_model_len=2048` y `gpu_memory_utilization=0.40`. Sube contexto solo despu?s de que `/models` responda.


In [ ]:
os.environ.setdefault("VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
VLLM_MODEL_CANDIDATES = [
    "unsloth/Qwen3-0.6B",
    "unsloth/Qwen3-4B-Instruct-2507",
    "unsloth/Qwen3-4B-Thinking-2507",
]

os.environ.setdefault("VLLM_MODEL", VLLM_MODEL_CANDIDATES[0])
os.environ.setdefault("VLLM_API_KEY", "EMPTY")
os.environ.setdefault("VLLM_MODE", "FAST")

vllm_env = {
    "VLLM_BASE_URL": os.getenv("VLLM_BASE_URL"),
    "VLLM_MODEL": os.getenv("VLLM_MODEL"),
    "VLLM_MODE": os.getenv("VLLM_MODE"),
    "VLLM_API_KEY_configured": bool(os.getenv("VLLM_API_KEY")),
}

toolkit.show(vllm_env, title="Configuracion vLLM segura")

scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=4, max_turns=4, max_concurrency=1)
vllm_runtime = toolkit.runtime(provider="vllm-runtime", scheduler=scheduler)
auto_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)

toolkit.show(vllm_runtime.describe(), title="vLLM runtime - describe")
toolkit.show(auto_runtime.describe(), title="Auto runtime - describe")


## 3) Unsloth 4-bit fast-inference smoke opcional

Este smoke es para comprobar la ruta Unsloth de carga local en 4-bit y fast inference. No levanta el endpoint vLLM usado por Agentic Systems; solo valida que Unsloth puede cargar el modelo en el entorno actual.

Por default est? apagado porque consume GPU.


In [ ]:
RUN_UNSLOTH_4BIT_SMOKE = False

if RUN_UNSLOTH_4BIT_SMOKE:
    from unsloth import FastLanguageModel
    import torch

    model_name = os.getenv("VLLM_MODEL", "unsloth/Qwen3-0.6B")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=2048,
        load_in_4bit=True,
        fast_inference=True,
    )
    FastLanguageModel.for_inference(model)
    toolkit.show({"status": "ok", "model": model_name, "load_in_4bit": True, "fast_inference": True}, title="Unsloth 4-bit smoke")
else:
    toolkit.show({"status": "skipped", "reason": "RUN_UNSLOTH_4BIT_SMOKE=False"}, title="Unsloth 4-bit smoke")


## 4) Levantar vLLM server opcional

Esta es la capa de infraestructura. Sigue la receta funcional tipo Qwen:

1. limpiar procesos vLLM y cache CUDA;
2. resolver perfil `FAST/MEDIUM/POWER`;
3. arrancar `vllm serve` como proceso background;
4. esperar `/v1/models`;
5. si falla, mostrar `vllm_server.log`.

El comando principal usa `vllm serve`, que es la ruta oficial del servidor OpenAI-compatible de vLLM. El endpoint resultante es lo que consume Agentic Systems.


In [ ]:
@dataclass
class VllmServerConfig:
    model: str
    served_model_name: str
    host: str = "127.0.0.1"
    port: int = 8000
    gpu_memory_utilization: float = 0.40
    max_model_len: int = 2048
    max_num_seqs: int = 4
    tool_call_parser: str = "hermes"
    reasoning_parser: str = "qwen3"
    enable_tool_choice: bool = True


def vllm_profile(mode: str, model: str) -> VllmServerConfig:
    mode = mode.upper().strip()
    if mode == "POWER":
        return VllmServerConfig(model=model, served_model_name=model, gpu_memory_utilization=0.90, max_model_len=32768, max_num_seqs=1)
    if mode == "MEDIUM":
        return VllmServerConfig(model=model, served_model_name=model, gpu_memory_utilization=0.55, max_model_len=4096, max_num_seqs=4)
    return VllmServerConfig(model=model, served_model_name=model, gpu_memory_utilization=0.40, max_model_len=2048, max_num_seqs=4)


def cleanup_gpu_and_vllm_processes() -> None:
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    os.system('pkill -f "vllm.entrypoints.openai.api_server" || true')
    os.system('pkill -f "vllm serve" || true')
    os.system('pkill -f "vllm" || true')


def url_open_no_proxy(url: str, *, timeout: float = 3.0) -> dict:
    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
    request = Request(url, headers={"Authorization": f"Bearer {os.getenv('VLLM_API_KEY', 'EMPTY')}"})
    with opener.open(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def tail_text(path: str, *, max_chars: int = 8000) -> str:
    try:
        with open(path, "r", encoding="utf-8", errors="replace") as handle:
            return handle.read()[-max_chars:]
    except FileNotFoundError:
        return ""


def wait_for_models(base_url: str, process: subprocess.Popen | None = None, *, log_path: str = "vllm_server.log", timeout_s: int = 300) -> dict:
    models_url = base_url.rstrip("/") + "/models"
    deadline = time.time() + timeout_s
    last_error = None
    while time.time() < deadline:
        if process is not None and process.poll() is not None:
            return {"status": "failed", "models_url": models_url, "returncode": process.returncode, "log_tail": tail_text(log_path)}
        try:
            payload = url_open_no_proxy(models_url, timeout=3)
            return {"status": "ok", "models_url": models_url, "response": payload}
        except Exception as exc:
            last_error = str(exc)
            time.sleep(3)
    return {"status": "timeout", "models_url": models_url, "reason": last_error, "log_tail": tail_text(log_path)}


def launch_vllm_server(config: VllmServerConfig) -> dict:
    log_path = "vllm_server.log"
    base_url = f"http://{config.host}:{config.port}/v1"
    existing = wait_for_models(base_url, timeout_s=3)
    if existing["status"] == "ok":
        return {"status": "already_running", "base_url": base_url, "health": existing}

    cleanup_gpu_and_vllm_processes()
    vllm_bin = shutil.which("vllm") or "vllm"
    cmd = [
        vllm_bin,
        "serve",
        config.model,
        "--host",
        config.host,
        "--port",
        str(config.port),
        "--served-model-name",
        config.served_model_name,
        "--gpu-memory-utilization",
        str(config.gpu_memory_utilization),
        "--max-model-len",
        str(config.max_model_len),
        "--max-num-seqs",
        str(config.max_num_seqs),
    ]
    if config.enable_tool_choice:
        cmd += ["--enable-auto-tool-choice", "--tool-call-parser", config.tool_call_parser]
    if config.reasoning_parser:
        cmd += ["--reasoning-parser", config.reasoning_parser]

    process = subprocess.Popen(cmd, stdout=open(log_path, "w", encoding="utf-8"), stderr=subprocess.STDOUT)
    health = wait_for_models(base_url, process, log_path=log_path, timeout_s=300)
    if health["status"] == "ok":
        os.environ["VLLM_BASE_URL"] = base_url
        os.environ["VLLM_MODEL"] = config.served_model_name
        return {"status": "started", "pid": process.pid, "base_url": base_url, "cmd": cmd, "log_path": log_path, "health": health}
    return {"status": "starting_or_failed", "pid": process.pid, "returncode": process.poll(), "base_url": base_url, "cmd": cmd, "log_path": log_path, "health": health}


RUN_VLLM_SERVER_SETUP = False

if RUN_VLLM_SERVER_SETUP:
    config = vllm_profile(os.getenv("VLLM_MODE", "FAST"), os.getenv("VLLM_MODEL", "unsloth/Qwen3-0.6B"))
    server_status = launch_vllm_server(config)
else:
    server_status = {"status": "skipped", "reason": "RUN_VLLM_SERVER_SETUP=False"}

toolkit.show(server_status, title="vLLM server launch opcional")


## 5) Health check del endpoint OpenAI-compatible

Esta celda es el gate real. Si `/models` no responde, no seguimos con inferencia ni con Agentic Systems.


In [ ]:
health = wait_for_models(os.getenv("VLLM_BASE_URL", "http://127.0.0.1:8000/v1"), timeout_s=5)
VLLM_SERVER_AVAILABLE = health["status"] == "ok"
toolkit.show(health, title="vLLM server health")


## 6) Smoke directo con OpenAI SDK

Antes de culpar o validar Agentic Systems, probamos el endpoint con el SDK OpenAI-compatible.


In [ ]:
if not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="OpenAI SDK smoke")
else:
    from openai import OpenAI

    client = OpenAI(base_url=os.getenv("VLLM_BASE_URL"), api_key=os.getenv("VLLM_API_KEY", "EMPTY"))
    response = client.chat.completions.create(
        model=os.getenv("VLLM_MODEL", "unsloth/Qwen3-0.6B"),
        messages=[{"role": "user", "content": "Responde solo: ok"}],
        temperature=0.0,
        max_tokens=16,
    )
    toolkit.show({"status": "ok", "text": response.choices[0].message.content}, title="OpenAI SDK smoke")


## 7) Agentic Systems smoke con `vllm-runtime`

Ahora s? probamos la API can?nica: tool, policy, agent, runtime y output humano.


In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos enteros."""
    return {"result": a + b}

policy = toolkit.RunPolicy(max_turns=4, max_tool_calls=2, temperature=0.0, tool_choice="auto", repair=True, max_repairs=1, trace="compact", strict=True)

if not VLLM_SERVER_AVAILABLE:
    toolkit.show({"status": "skipped", "reason": "Servidor vLLM no disponible."}, title="Tool smoke vLLM")
    result = None
else:
    system = toolkit.AgenticSystem(runtime=vllm_runtime)
    agent = system.agent(
        name="vllm_calculator",
        instructions="Usa la tool sumar para sumar 10 y 20. Devuelve un JSON compacto con result.",
        tools=[sumar],
    )
    result = agent.run("Suma 10 y 20 usando la tool sumar.", policy=policy)
    toolkit.human_result(result)


## 8) Roadmap futuro: Unsloth -> fine-tuning -> export -> vLLM -> Agentic Systems

Este notebook solo cubre inferencia. La ruta futura queda documentada, no estabilizada como API:

```text
Fine-tuning / continued pretraining
    -> Unsloth
    -> export merged_16bit, LoRA o merged_4bit segun objetivo
    -> vLLM server OpenAI-compatible
    -> Agentic Systems runtime(provider="vllm-runtime")
    -> Skills / Agents / Systems / Evals
```

Regla para produccion: entrenar/exportar modelos es responsabilidad de Unsloth/MLOps; Agentic Systems consume endpoints auditables y ejecuta herramientas, skills, agentes, sistemas y evaluaciones.


In [ ]:
api_coverage = [
    {"api": "toolkit.runtime(provider='vllm-runtime')", "description": "Declara vLLM como provider canonico OpenAI-compatible."},
    {"api": "toolkit.runtime(provider='auto')", "description": "Selecciona vLLM automaticamente cuando VLLM_BASE_URL esta configurado."},
    {"api": "RuntimeConfig.describe", "description": "Muestra resolucion y configuracion segura."},
    {"api": "toolkit.scheduler", "description": "Declara limites de ejecucion."},
    {"api": "toolkit.RunPolicy", "description": "Controla turns, tools, temperatura y reparacion."},
    {"api": "toolkit.tool", "description": "Define tools ejecutables por el agente."},
    {"api": "toolkit.AgenticSystem", "description": "Agrupa runtime, tools y agentes nativos."},
    {"api": "toolkit.human_result", "description": "Renderiza resultados humanos estables."},
]

toolkit.show({"notebook": "00_runtime_vllm_provider_api.ipynb", "api_coverage": api_coverage}, title="Cobertura API del notebook")
